In [4]:
import json
import re
import csv

# ── Load the JSON ─────────────────────────────────────────────────────────────
with open(r'C:\Users\nasee\Downloads\Gym.json', encoding='utf-8') as f:
    data = json.load(f)

# ── Exercise name aliases ─────────────────────────────────────────────────────
ALIASES = {
    "Inclined Smith Bench Press": "Inclined Smith Bench Press",
    "Inclined Smith Bench": "Inclined Smith Bench Press",
    "Inclined Smith Machine Bench Press": "Inclined Smith Bench Press",
    "Inclined Smith Machine Bench": "Inclined Smith Bench Press",
    "Inclined Smith Machine Press": "Inclined Smith Bench Press",
    "Inclined Smith Machine": "Inclined Smith Bench Press",
    "Inclined Smich Bench Press": "Inclined Smith Bench Press",
    "Inclined Smich Bench (Steep)": "Inclined Smith Bench Press",
    "Inclined Smitch Bench Press": "Inclined Smith Bench Press",
    "Inclined Smich Machine Bench (Old Mech)": "Inclined Smith Bench Press",
    "Inclined Smith Machine Benxh": "Inclined Smith Bench Press",
    "Incline Smith Bench": "Inclined Smith Bench Press",
    "Smith Bench Press": "Inclined Smith Bench Press",
    "Smith Bench": "Inclined Smith Bench Press",
    "Smith Inclined Bench Press": "Inclined Smith Bench Press",
    "Smith Inclined Bench": "Inclined Smith Bench Press",
    "Smith Inclined Chest": "Inclined Smith Bench Press",
    "Smith Machine Bench Press": "Inclined Smith Bench Press",
    "Easy Smitch Inclined Bench": "Inclined Smith Bench Press",
    "Inclined Smith Bench Press (Easy)": "Inclined Smith Bench Press",
    "Smich Inclined Bench Press": "Inclined Smith Bench Press",
    "Smitch Machine Inclined Bench Press": "Inclined Smith Bench Press",
    "Incined Smith Bench Press": "Inclined Smith Bench Press",
    "Inclined Bench Press Machine": "Inclined Machine Bench Press",
    "Inclined Chest Press Machine": "Inclined Machine Bench Press",
    "Inclined Machine Bench Press": "Inclined Machine Bench Press",
    "Inclined Machine Chest Press": "Inclined Machine Bench Press",
    "Inclined Machine Press": "Inclined Machine Bench Press",
    "Machine Inclined Bench": "Inclined Machine Bench Press",
    "Machine Inclined Bench Press": "Inclined Machine Bench Press",
    "Machine Inclined Chest": "Inclined Machine Bench Press",
    "Machine Inclined Chest Press": "Inclined Machine Bench Press",
    "Machine Inclined Press": "Inclined Machine Bench Press",
    "Machine Bench Press": "Inclined Machine Bench Press",
    "Inclined Bench Press": "Inclined Machine Bench Press",
    "Inclined Dumbell Bench": "Inclined Dumbbell Bench Press",
    "Lat Pull Down": "Lat Pull Down",
    "Lat Pull Down (Crap Machine)": "Lat Pull Down",
    "Lat Pull Down (Wide Grip)": "Lat Pull Down (Wide Grip)",
    "Lat Pushdown": "Lat Pull Down",
    "Preacher Curl": "Preacher Curl",
    "Machine Preacher Curl": "Preacher Curl",
    "Machine Preacher Curl :": "Preacher Curl",
    "Preacher Curl Machine": "Preacher Curl",
    "Preacher Machine Curl": "Preacher Curl",
    "Preacher": "Preacher Curl",
    "Back Row": "Cable Row",
    "Backrow": "Cable Row",
    "Back Cable Pull": "Cable Row",
    "Cable Back Row": "Cable Row",
    "Cable Row": "Cable Row",
    "Machine Back Row": "Cable Row",
    "Side Raise": "Side Delt Raise",
    "Side Shoulder Raise": "Side Delt Raise",
    "Side Shoulder Cable": "Side Delt Raise (Cable)",
    "Cable Side Raise": "Side Delt Raise (Cable)",
    "Shoulder Press": "Shoulder Press",
    "Machine Shoulder Press": "Shoulder Press",
    "Tricep Push Down": "Tricep Push Down",
    "Tricep Pushdown": "Tricep Push Down",
    "Chest Fly": "Machine Chest Fly",
    "Chest Fly (Max Resistance)": "Machine Chest Fly",
    "Machine Chest Fly": "Machine Chest Fly",
    "Pectoral Machine": "Machine Chest Fly",
    "Smith Squats": "Smith Squats",
    "Smuth Squat": "Smith Squats",
    "Reverse Fly Rear Delt": "Rear Delt Fly",
}

def normalise_exercise(name):
    return ALIASES.get(name, name)

# ── Parse the data ────────────────────────────────────────────────────────────
text = data['textContent']
lines = text.split('\n')

date_re = re.compile(r'^\d{1,2}/\d{1,2}/\d{2,4}$')
set_re  = re.compile(r'(\d+\.?\d*)\s*kg\s*[-–]\s*(\d+)', re.IGNORECASE)

rows = []
current_date     = None
current_exercise = None
set_number       = 0

for line in lines:
    line = line.strip()
    if not line:
        continue

    if date_re.match(line):
        d, m, y = line.split('/')
        if len(y) == 2:
            y = '20' + y
        current_date     = f"{y}-{m.zfill(2)}-{d.zfill(2)}"
        current_exercise = None
        set_number       = 0

    elif set_re.search(line):
        match  = set_re.search(line)
        weight = float(match.group(1))
        reps   = int(match.group(2))
        set_number += 1
        rows.append({
            'date':       current_date,
            'exercise':   normalise_exercise(current_exercise or 'Unknown'),
            'set_number': set_number,
            'weight_kg':  weight,
            'reps':       reps,
            'volume_kg':  round(weight * reps, 2),
        })

    else:
        current_exercise = line.title().strip()
        set_number       = 0

# ── Save to CSV ───────────────────────────────────────────────────────────────
out_path = r'C:\Users\nasee\Downloads\gym_data.csv'
fields   = ['date', 'exercise', 'set_number', 'weight_kg', 'reps', 'volume_kg']

with open(out_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()
    writer.writerows(rows)

# ── Summary ───────────────────────────────────────────────────────────────────
exercises = sorted(set(r['exercise'] for r in rows))
dates     = sorted(set(r['date'] for r in rows))

print(f"✅  {len(rows)} sets across {len(dates)} sessions")
print(f"✅  {len(exercises)} unique exercises")
print(f"✅  Saved to {out_path}")
print(f"\nDate range: {dates[0]}  →  {dates[-1]}")

✅  977 sets across 63 sessions
✅  19 unique exercises
✅  Saved to C:\Users\nasee\Downloads\gym_data.csv

Date range: 2025-09-16  →  2026-09-13


In [6]:
import pandas as pd

df = pd.read_csv(r'C:\Users\nasee\Downloads\gym_data.csv')
df

,date,exercise,set_number,weight_kg,reps,volume_kg
0,2026-05-09,Inclined Smith Bench Press,1,40.0,8,320.0
1,2026-05-09,Inclined Smith Bench Press,2,60.0,6,360.0
2,2026-05-09,Inclined Smith Bench Press,3,60.0,5,300.0
3,2026-05-09,Inclined Smith Bench Press,4,60.0,3,180.0
4,2026-05-09,Lat Pull Down,1,70.0,12,840.0
...,...,...,...,...,...,...
972,2025-10-25,Machine Chest Fly,3,68.0,6,408.0
973,2025-10-25,Tricep Push Down,1,30.0,8,240.0
974,2025-10-25,Tricep Push Down,2,35.0,4,140.0
975,2025-10-25,Tricep Push Down,3,32.5,5,162.5
